# 🎭 VIBE-X Interactive Tutorial

**Vector-Integrated Binary Extension — "Encode Once, Query Infinitely."**

This notebook provides a hands-on introduction to the VIBE-X protocol for encoding emotional metadata into text.

[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.17228992.svg)](https://doi.org/10.5281/zenodo.17228992)
[![PyPI version](https://img.shields.io/pypi/v/vibex-protocol.svg)](https://pypi.org/project/vibex-protocol/)

---

## 📚 Table of Contents

1. [Installation & Setup](#installation)
2. [Basic Encoding](#basic-encoding)
3. [Basic Decoding](#basic-decoding)
4. [Multiple Annotations](#multiple-annotations)
5. [Understanding the 14-bit MetaBlock](#metablock)
6. [Real-World Example: Social Media Analysis](#social-media)
7. [Performance Benchmarks](#benchmarks)
8. [Advanced: Custom Tokenizers](#advanced)

## 1. Installation & Setup

First, let's install the VIBE-X protocol library:

In [ ]:
# Install from PyPI
# !pip install vibex-protocol

# Or install from source (if in repo)
# !pip install -e ..

# Import required modules
from vibex import InlineEncoder, InlineDecoder, SentimentAnnotation, Tokenizer, MetaBlock
import time

## 2. Basic Encoding

Let's start with a simple example: encoding emotional metadata for a positive sentence.

In [ ]:
# Initialize encoder with tokenizer
encoder = InlineEncoder(Tokenizer())

# Sample text
text = "The movie was absolutely amazing"

# First, let's see how the text is tokenized
tokenizer = Tokenizer()
tokens = tokenizer.tokenize(text)
print("Tokens:", tokens)
print("Token indices:", {i: token for i, token in enumerate(tokens)})

In [ ]:
# Create an annotation for "absolutely amazing" (tokens 3-4)
annotation = SentimentAnnotation(
    anchor=3,        # Starting at "absolutely"
    length=2,        # Covers 2 tokens: "absolutely amazing"
    polarity=2,      # Positive (0=Neutral, 1=Negative, 2=Positive, 3=Ironic)
    intensity=6,     # Strong emotion (0-7 scale)
    context=0,       # Literal/Static (0=Static, 1=Dynamic)
    emotion=1,       # Joy (emotion class)
    reserved=0  # Not an emergency
)

# Encode the text
encoded_text = encoder.encode(text, [annotation])

print("Original text:", text)
print("Encoded text:", encoded_text)
print("\nNotice the Unicode markers (invisible in some displays) around the annotated span!")

## 3. Basic Decoding

Now let's decode the encoded text to extract the metadata.

In [ ]:
# Initialize decoder
decoder = InlineDecoder(Tokenizer())

# Decode the text
decoded = decoder.decode(encoded_text)

print("Clean text (markers removed):", decoded.clean_text)
print("Clean tokens:", decoded.clean_tokens)
print("\nExtracted MetaBlocks:")

for i, block in enumerate(decoded.blocks):
    print(f"\nBlock {i+1}:")
    print(f"  Hex encoding: {block.block.to_hex()}")
    print(f"  Integer value: {block.block.to_int()}")
    print(f"  Anchor: {block.anchor}")
    print(f"  Span: {block.span} tokens")
    print(f"  Polarity: {['Neutral', 'Negative', 'Positive', 'Ironic'][block.block.polarity]}")
    print(f"  Intensity: {block.block.intensity}/7")
    print(f"  Context: {'Dynamic' if block.block.context else 'Static'}")
    print(f"  Emotion: {['Neutral', 'Joy', 'Trust', 'Fear', 'Surprise', 'Sadness', 'Disgust', 'Anger'][block.block.emotion]}")
    print(f"  Reserved: {block.block.reserved}")

## 4. Multiple Annotations

VIBE-X supports multiple annotations in the same text, even overlapping spans.

In [ ]:
# Complex sentence with mixed emotions
text = "I loved the performance but the ending felt rushed"

# Tokenize to understand indices
tokens = tokenizer.tokenize(text)
print("Tokens:", {i: token for i, token in enumerate(tokens)})

# Create multiple annotations
annotations = [
    SentimentAnnotation(
        anchor=1,       # "loved"
        length=1,
        polarity=2,     # Positive
        intensity=6,
        context=0,
        emotion=1       # Joy
    ),
    SentimentAnnotation(
        anchor=7,       # "felt"
        length=2,       # "felt rushed"
        polarity=1,     # Negative
        intensity=4,
        context=1,      # Context-dependent
        emotion=5       # Sadness
    ),
]

# Encode
encoded = encoder.encode(text, annotations)
print("\nEncoded:", encoded)

# Decode
decoded = decoder.decode(encoded)
print("\nFound", len(decoded.blocks), "metadata blocks:")
for i, block in enumerate(decoded.blocks):
    emotion_name = ['Neutral', 'Joy', 'Trust', 'Fear', 'Surprise', 'Sadness', 'Disgust', 'Anger'][block.block.emotion]
    polarity_name = ['Neutral', 'Negative', 'Positive', 'Ironic'][block.block.polarity]
    print(f"  Block {i+1}: {polarity_name} {emotion_name} (intensity {block.block.intensity}/7)")

## 5. Understanding the 14-bit MetaBlock

Let's examine the binary structure of a MetaBlock in detail.

In [ ]:
# Create a MetaBlock directly
meta = MetaBlock(
    has_span=True,
    span=3,          # 3 tokens
    polarity=2,      # Positive
    intensity=7,     # Maximum intensity
    context=1,       # Dynamic
    emotion=1,       # Joy
    reserved=1   # Emergency flag set
)

# Convert to different representations
int_value = meta.to_int()
hex_value = meta.to_hex()
binary_value = bin(int_value)[2:].zfill(14)

print("MetaBlock Representations:")
print(f"  Integer: {int_value}")
print(f"  Hexadecimal: {hex_value}")
print(f"  Binary (14-bit): {binary_value}")
print("\nBit Layout (SPICE-R):")
print(f"  Has_SPAN (1 bit):      {binary_value[0]}")
print(f"  Span length (3 bits):  {binary_value[1:4]}  = {int(binary_value[1:4], 2)}")
print(f"  Polarity (2 bits):     {binary_value[4:6]}  = {int(binary_value[4:6], 2)} (Positive)")
print(f"  Intensity (3 bits):    {binary_value[6:9]}  = {int(binary_value[6:9], 2)}")
print(f"  Context (1 bit):       {binary_value[9]}    = {int(binary_value[9], 2)} (Dynamic)")
print(f"  Emotion (3 bits):      {binary_value[10:13]} = {int(binary_value[10:13], 2)} (Joy)")
print(f"  Emergency (1 bit):     {binary_value[13]}   = {int(binary_value[13], 2)} (True)")

# Round-trip test
restored = MetaBlock.from_hex(hex_value)
print("\nRound-trip test:", meta == restored)

## 6. Real-World Example: Social Media Analysis

Let's simulate analyzing social media posts.

In [ ]:
# Sample social media posts
posts = [
    "Just got promoted at work! Best day ever!",
    "This traffic is absolutely horrible and frustrating.",
    "The weather is nice today.",
    "Oh great, another meeting. Just what I needed.",  # Sarcasm
    "URGENT: Need help immediately!",
]

# Simulate sentiment analysis (in real world, this would be ML model output)
annotations_per_post = [
    # Post 1: Very positive
    [SentimentAnnotation(anchor=2, length=1, polarity=2, intensity=7, context=0, emotion=1)],
    # Post 2: Very negative
    [SentimentAnnotation(anchor=3, length=3, polarity=1, intensity=6, context=0, emotion=6)],
    # Post 3: Neutral
    [SentimentAnnotation(anchor=3, length=1, polarity=0, intensity=2, context=0, emotion=0)],
    # Post 4: Ironic/Sarcastic
    [SentimentAnnotation(anchor=1, length=1, polarity=3, intensity=5, context=1, emotion=7)],
    # Post 5: Emergency
    [SentimentAnnotation(anchor=0, length=1, polarity=1, intensity=7, context=0, emotion=3, reserved=1)],
]

# Encode all posts
encoded_posts = []
for post, annotations in zip(posts, annotations_per_post):
    encoded = encoder.encode(post, annotations)
    encoded_posts.append(encoded)

print("Encoded Posts:")
for i, (original, encoded) in enumerate(zip(posts, encoded_posts)):
    print(f"\n{i+1}. {original}")
    print(f"   Encoded: {encoded}")

# Now simulate querying
print("\n" + "="*70)
print("QUERY EXAMPLES (Zero-cost after encoding!)")
print("="*70)

# Query 1: Find posts with reserved flag set (emergency)
print("\n🚨 Emergency posts:")
for i, encoded in enumerate(encoded_posts):
    decoded = decoder.decode(encoded)
    if any(block.block.reserved for block in decoded.blocks):
        print(f"  - Post {i+1}: {posts[i]}")

# Query 2: Find highly negative posts
print("\n😢 Highly negative posts (polarity=1, intensity>=5):")
for i, encoded in enumerate(encoded_posts):
    decoded = decoder.decode(encoded)
    for block in decoded.blocks:
        if block.block.polarity == 1 and block.block.intensity >= 5:
            print(f"  - Post {i+1}: {posts[i]}")
            break

# Query 3: Find sarcastic/ironic posts
print("\n😏 Sarcastic posts:")
for i, encoded in enumerate(encoded_posts):
    decoded = decoder.decode(encoded)
    if any(block.block.polarity == 3 for block in decoded.blocks):
        print(f"  - Post {i+1}: {posts[i]}")

# Query 4: Find joyful posts
print("\n😊 Joyful posts (emotion=1):")
for i, encoded in enumerate(encoded_posts):
    decoded = decoder.decode(encoded)
    if any(block.block.emotion == 1 for block in decoded.blocks):
        print(f"  - Post {i+1}: {posts[i]}")

## 7. Performance Benchmarks

Let's measure the performance of VIBE-X encoding and decoding.

In [ ]:
# Benchmark encoding
test_text = "The movie was absolutely amazing and I loved every minute of it"
test_annotation = SentimentAnnotation(anchor=3, length=2, polarity=2, intensity=6, context=0, emotion=1)

# Encoding benchmark
iterations = 10000
start = time.perf_counter()
for _ in range(iterations):
    encoded = encoder.encode(test_text, [test_annotation])
end = time.perf_counter()

encoding_time = (end - start) / iterations * 1_000_000  # microseconds
print(f"Encoding time: {encoding_time:.3f} µs per operation")

# Decoding benchmark
start = time.perf_counter()
for _ in range(iterations):
    decoded = decoder.decode(encoded)
end = time.perf_counter()

decoding_time = (end - start) / iterations * 1_000_000  # microseconds
print(f"Decoding time: {decoding_time:.3f} µs per operation")

# MetaBlock operations
meta = MetaBlock(has_span=True, span=2, polarity=2, intensity=6, context=0, emotion=1)
hex_val = meta.to_hex()

start = time.perf_counter()
for _ in range(iterations):
    restored = MetaBlock.from_hex(hex_val)
end = time.perf_counter()

metablock_time = (end - start) / iterations * 1_000_000  # microseconds
print(f"MetaBlock decode time: {metablock_time:.3f} µs per operation")

# Storage comparison
import json
import sys

# VIBE-X storage (14 bits = 1.75 bytes, but with hex encoding it's 4 chars = 4 bytes)
vibe_x_size = len(hex_val)  # bytes

# JSON equivalent
json_metadata = {
    "anchor": 3,
    "span": 2,
    "polarity": "positive",
    "intensity": 6,
    "context": "static",
    "emotion": "joy"
}
json_size = len(json.dumps(json_metadata))

print(f"\nStorage comparison:")
print(f"  VIBE-X (hex): {vibe_x_size} bytes")
print(f"  JSON: {json_size} bytes")
print(f"  Savings: {(1 - vibe_x_size/json_size) * 100:.1f}%")

# At scale
million_posts = 1_000_000
vibe_x_total = vibe_x_size * million_posts / (1024**2)  # MB
json_total = json_size * million_posts / (1024**2)  # MB

print(f"\nAt 1M posts scale:")
print(f"  VIBE-X: {vibe_x_total:.2f} MB")
print(f"  JSON: {json_total:.2f} MB")
print(f"  Saved: {json_total - vibe_x_total:.2f} MB")

## 8. Advanced: Error Handling

VIBE-X includes proper error handling for invalid inputs.

In [ ]:
from vibex.exceptions import MetaBlockEncodingError

text = "Short text"
tokens = tokenizer.tokenize(text)
print(f"Text has {len(tokens)} tokens: {tokens}")

# Try to annotate a token that doesn't exist
try:
    bad_annotation = SentimentAnnotation(
        anchor=50,  # Way out of bounds!
        length=1,
        polarity=2,
        intensity=5,
        context=0,
        emotion=1
    )
    encoder.encode(text, [bad_annotation])
except MetaBlockEncodingError as e:
    print(f"\n✅ Caught expected error: {e}")

# Try invalid hex
try:
    MetaBlock.from_hex("ZZZZ")
except ValueError as e:
    print(f"\n✅ Caught expected error: {e}")

## 🎯 Conclusion

You've learned:

1. ✅ How to encode emotional metadata with VIBE-X
2. ✅ How to decode and extract metadata
3. ✅ Working with multiple annotations
4. ✅ Understanding the 14-bit MetaBlock structure
5. ✅ Real-world application patterns
6. ✅ Performance characteristics
7. ✅ Error handling

### Next Steps

- 📖 Read the [full documentation](https://doi.org/10.5281/zenodo.17228992)
- 💻 Try the [Streamlit demo](../demo/streamlit_app.py)
- 🌟 Star the [GitHub repository](https://github.com/vibexcode/vibe-x)
- 🤝 Contribute or report issues

---

**VIBE-X Protocol v1.0.0** | MIT License | Created by Uğur Kandemiş